<a href="https://colab.research.google.com/github/angadpreeta/AI_ML_Project/blob/main/FAQ_LLM_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning DistilGPT-2 for E-Commerce Customer Support FAQs

**Assignment: Fine-Tuning a Small LLM for Customer Support FAQs**

This notebook walks through the full pipeline:
1. Environment setup
2. Dataset preparation
3. LoRA fine-tuning of DistilGPT-2
4. Evaluation (BLEU, ROUGE, cosine similarity)
5. Inference demo



## Step 1: Install Dependencies

In [1]:
# Install all required libraries
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3 datasets==2.18.0
!pip install -q rouge-score nltk sentence-transformers scikit-learn gradio
!pip install -q gradio bitsandbytes
print('Installation complete')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 20

## Step 2: Project from Drive & Verify GPU

In [28]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# Create project structure
os.makedirs('/content/data/processed', exist_ok=True)
os.makedirs('/content/outputs/model', exist_ok=True)
os.makedirs('/content/outputs/evaluation', exist_ok=True)
os.makedirs('/content/scripts', exist_ok=True)

# Copy files from Drive to working directory
src = '/content/drive/MyDrive/files'

shutil.copy(f'{src}/ecommerce_faqs.jsonl', '/content/data/ecommerce_faqs.jsonl')
for script in ['prepare_dataset.py', 'train.py', 'evaluate.py', 'inference.py','FAQ_LLM_Finetuning.ipynb']:
    shutil.copy(f'{src}/{script}', f'/content/scripts/{script}')

print("✓ All files copied to /content")

CUDA available: True
GPU: Tesla T4
Memory: 15.64 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ All files copied to /content


## Step 3: Dataset Preparation

In [29]:
# Run dataset preparation script
# Make sure ecommerce_faqs.jsonl is in data/ folder
%cd /content
!python scripts/prepare_dataset.py

# Verify files were created
import json
for split in ['train', 'val', 'test']:
    with open(f'data/processed/{split}.jsonl') as f:
        count = sum(1 for line in f if line.strip())
    print(f'{split}: {count} examples')

/content
  E-Commerce FAQ Dataset Preparation

[1] Loading data from: data/ecommerce_faqs.jsonl
  Loaded 731 raw records

[2] Validating and cleaning records...
  731/731 records passed validation

[3] Formatting records for training...

[4] Dataset statistics:

  [Full Dataset] 731 records
    Avg prompt length  : 42.7 chars
    Avg response length: 204.4 chars
    Max prompt length  : 80 chars
    Max response length: 268 chars

[5] Splitting dataset (80% train / 10% val / 10% test)...

[6] Saving split datasets...
  Saved 584 records to data/processed/train.jsonl
  Saved 73 records to data/processed/val.jsonl
  Saved 74 records to data/processed/test.jsonl
  Saved 74 records to data/processed/test_prompts.jsonl

[7] Sample formatted training record:
--------------------------------------------------
<|user|>
How do I track my order?
<|assistant|>
You can track your order by going to 'My Orders' in your account and clicking on the order number to see real-time tracking updates.<|end|

In [30]:
# Preview a training sample
with open('data/processed/train.jsonl') as f:
    sample = json.loads(f.readline())
print('=== Sample Training Record ===')
print(sample['text'])

=== Sample Training Record ===
<|user|>
Do you sell genuine products?
<|assistant|>
Yes, all products on our platform are genuine and sourced directly from authorised manufacturers, brand authorised sellers, or official distributors. We have a strict seller verification process.<|end|>


In [31]:
# Fix the train.py script - remove unsupported load_in_8bit argument for GPT2
fix = open('/content/scripts/train.py').read()

old = '''    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        load_in_8bit=load_in_8bit,
        device_map="auto" if load_in_8bit else None,
        torch_dtype=torch.float16 if load_in_8bit else torch.float32,
    )'''

new = '''    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
    )'''

fix = fix.replace(old, new)
open('/content/scripts/train.py', 'w').write(fix)
print("✓ train.py patched successfully")

✓ train.py patched successfully


In [32]:
fix = open('/content/scripts/train.py').read()

fix = fix.replace(
    'evaluation_strategy="epoch"',
    'eval_strategy="epoch"'
)

open('/content/scripts/train.py', 'w').write(fix)
print("✓ Patched evaluation_strategy → eval_strategy")

✓ Patched evaluation_strategy → eval_strategy


In [33]:
!pip install -q "accelerate>=1.1.0" "transformers[torch]"
print("✓ Done")

✓ Done


## Step 4: Fine-Tuning with LoRA

In [21]:
# Run fine-tuning
# Estimated time: ~10-15 minutes on T4 GPU for 5 epochs
!python scripts/train.py \
    --model_name distilgpt2 \
    --data_dir data/processed \
    --output_dir outputs/model \
    --epochs 6 \
    --batch_size 8 \
    --lr 5e-4 \
    --lora_r 8 \
    --lora_alpha 32 \
    --lora_dropout 0.1 \
    --use_lora \
    --max_length 256

2026-03-10 11:37:10,297 - INFO - ============================================================
2026-03-10 11:37:10,298 - INFO -   Starting FAQ LLM Fine-Tuning
2026-03-10 11:37:10,298 - INFO - ============================================================
2026-03-10 11:37:10,298 - INFO - Device: cuda
2026-03-10 11:37:10,298 - INFO - GPU: Tesla T4
2026-03-10 11:37:10,298 - INFO - GPU Memory: 15.64 GB
2026-03-10 11:37:10,298 - INFO - Loading tokenizer: distilgpt2
2026-03-10 11:37:10,448 - INFO - HTTP Request: HEAD https://huggingface.co/distilgpt2/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-10 11:37:10,530 - INFO - HTTP Request: HEAD https://huggingface.co/distilgpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-10 11:37:10,530 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-03-10 11:37:10,614 - INFO - HTTP Request: GET https://huggingface.co/api/models/distilgpt2/

In [22]:
# List saved checkpoints
import os
for root, dirs, files in os.walk('outputs/model'):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1e6
        print(f'{path}  ({size_mb:.1f} MB)')

outputs/model/training_config.json  (0.0 MB)
outputs/model/final/README.md  (0.0 MB)
outputs/model/final/tokenizer.json  (3.6 MB)
outputs/model/final/training_args.bin  (0.0 MB)
outputs/model/final/adapter_config.json  (0.0 MB)
outputs/model/final/tokenizer_config.json  (0.0 MB)
outputs/model/final/adapter_model.safetensors  (309.4 MB)
outputs/model/checkpoint-185/scheduler.pt  (0.0 MB)
outputs/model/checkpoint-185/README.md  (0.0 MB)
outputs/model/checkpoint-185/optimizer.pt  (1.2 MB)
outputs/model/checkpoint-185/training_args.bin  (0.0 MB)
outputs/model/checkpoint-185/scaler.pt  (0.0 MB)
outputs/model/checkpoint-185/rng_state.pth  (0.0 MB)
outputs/model/checkpoint-185/trainer_state.json  (0.0 MB)
outputs/model/checkpoint-185/adapter_config.json  (0.0 MB)
outputs/model/checkpoint-185/adapter_model.safetensors  (309.4 MB)
outputs/model/checkpoint-222/scheduler.pt  (0.0 MB)
outputs/model/checkpoint-222/README.md  (0.0 MB)
outputs/model/checkpoint-222/optimizer.pt  (1.2 MB)
outputs/model

## Step 5: Evaluation

In [23]:
# Run evaluation
!python scripts/evaluate.py \
    --model_path outputs/model/final \
    --data_dir data/processed \
    --output_dir outputs/evaluation

  FAQ LLM Evaluation

Test set size: 74 (using first 10 for manual comparison)

[1] Loading tokenizer...

[2] Loading base model (distilgpt2)...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 76/76 [00:00<00:00, 7263.36it/s]
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`

[3] Loading fine-tuned model from: outputs/model/final
Loading weights: 100% 76/76 [00:00<00:00, 22202.90it/s]

[4] Generating answers...
  Query 1/74: How is the buyback value determined?...
  Query 2/74: What is your response time to emails?...
  Query 3/74: Do you offer senior citizen discounts?...
  Query 4/74: Do you follow BIS guidelines for product safety?...
  Query 5/74: How does your platform handle language barriers fo...
  Query 6/74: What shipping carriers can I use as a self-s

In [15]:
# Display evaluation results
with open('outputs/evaluation/evaluation_results.json') as f:
    results = json.load(f)

print('=== Automated Metrics ===')
metrics = results['metrics']
print(f'{"Metric":<20} {"Base Model":>15} {"Fine-tuned":>15}')
print('-' * 50)
for key in ['bleu', 'rouge1', 'rouge2', 'rougeL', 'cosine_sim']:
    b = metrics['base'].get(key)
    f = metrics['finetuned'].get(key)
    bstr = f'{b:.4f}' if b else 'N/A'
    fstr = f'{f:.4f}' if f else 'N/A'
    print(f'{key:<20} {bstr:>15} {fstr:>15}')

=== Automated Metrics ===
Metric                    Base Model      Fine-tuned
--------------------------------------------------
bleu                          0.0060          0.0087
rouge1                        0.1505          0.2131
rouge2                        0.0137          0.0258
rougeL                        0.0929          0.1250
cosine_sim                    0.3406          0.4983


## Step 6: Inference Demo

In [26]:
# Quick inference demo (without Gradio)
from scripts.inference import FAQBot

bot = FAQBot(model_path='outputs/model/final')

test_queries = [
    "Can you show me the upcoming offers?",
]

print('=== Fine-tuned Model Inference ===')
for query in test_queries:
    answer = bot.answer(query)
    print(f'Q: {query}')
    print(f'A: {answer}')
    print()

[FAQBot] Loading model from: outputs/model/final
[FAQBot] Device: cuda
[FAQBot] Detected LoRA checkpoint — loading base model + adapters


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[FAQBot] Model ready!
=== Fine-tuned Model Inference ===
Q: Can you show me the upcoming offers?
A: Yes, we offer new deals. In this case our e-commerce platform is available on select markets and features such as Shopify or Google+ channels for delivery details to your customers via direct mail (in Indian).We also provide a complete list of current offerings that will be sold at an official shop! You can view full listings here: We invite sellers to join us in making purchases with minimal hassle. The most common 'deals' are sponsored by brands like Baidu, Naira & HMDs.purchase_with=true , Best Buy's Deals page .cancelable = true , Topshop products, exclusive



In [27]:
# Launch Gradio UI (creates a public share link)
!python scripts/inference.py --gradio --model_path outputs/model/final

[FAQBot] Loading model from: outputs/model/final
[FAQBot] Device: cuda
[FAQBot] Detected LoRA checkpoint — loading base model + adapters
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 76/76 [00:00<00:00, 19555.06it/s]
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
[FAQBot] Model ready!
/content/scripts/inference.py:181: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="E-Commerce FAQ Bot", theme=gr.themes.Soft()) as demo:

Launching Gradio UI on port 7860...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://96e106a33505c080dc.gradio.live

This share link expires in 1 week. For free permanent hosting 

## Step 7: Download Model Checkpoint

In [19]:
# Zip and download the final model
!zip -r faq_model_final.zip outputs/model/final/
from google.colab import files
files.download('faq_model_final.zip')

  adding: outputs/model/final/ (stored 0%)
  adding: outputs/model/final/README.md (deflated 66%)
  adding: outputs/model/final/tokenizer.json (deflated 82%)
  adding: outputs/model/final/training_args.bin (deflated 53%)
  adding: outputs/model/final/adapter_config.json (deflated 52%)
  adding: outputs/model/final/tokenizer_config.json (deflated 50%)
  adding: outputs/model/final/adapter_model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
!pip install -q huggingface_hub
from huggingface_hub import login
login()  # paste your HF token from https://huggingface.co/settings/tokens


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
